In [147]:
import pandas as pd 
import numpy as np
import re
from pathlib import Path

import ipywidgets as widgets
import plotly.graph_objects as go
from IPython.display import display


In [148]:
sample_catalog_df = pd.read_csv('sample_sweetcat.csv')
best_rm_path = Path('observations/Best_RM')


def normalize_star_name(star_name):
    return re.sub(r'[^a-z0-9]', '', str(star_name).lower())


observations_df = pd.DataFrame(
    [
        {
            'observation_file': obs_path.name,
            'star_key': normalize_star_name(
                re.sub(r'_esp.*$', '', obs_path.stem, flags=re.IGNORECASE)
            ),
        }
        for obs_path in sorted(best_rm_path.glob('*.rdb'))
    ]
)
sample_catalog_with_keys = sample_catalog_df.assign(
    star_key=sample_catalog_df['star'].map(normalize_star_name)
)

sample_df = observations_df.merge(
    sample_catalog_with_keys,
    on='star_key',
    how='left',
).drop(columns='star_key')
sample_df = sample_df[['observation_file', *sample_catalog_df.columns]]

sample_df.head(2)

,observation_file,star,Name,gaia_dr3,Teff,eTeff,Logg,eLogg,[Fe/H],e[Fe/H],...,Gmag,Plx,Distance,Mass_t,Radius_t,SWFlag,Reference,Link,Update,Database
0,HD189733_esp19_3.rdb,HD189733,HD 189733,1827242816201846144,4969.0,48.0,4.3,0.12,-0.08,0.03,...,7.4284,50.5668,19.775821,0.751097,0.786858,1,Sousa et al. 2021,https://ui.adsabs.harvard.edu/abs/2021arXiv210...,2021-01-01,EU;NASA
1,HD189733_esp19_4.rdb,HD189733,HD 189733,1827242816201846144,4969.0,48.0,4.3,0.12,-0.08,0.03,...,7.4284,50.5668,19.775821,0.751097,0.786858,1,Sousa et al. 2021,https://ui.adsabs.harvard.edu/abs/2021arXiv210...,2021-01-01,EU;NASA


In [149]:
file_path = best_rm_path


def read_rdb_file(file_path):
    rdb_df = pd.read_csv(file_path, sep='\t', skiprows=[1])

    for column in ['rjd', 'vrad', 'svrad']:
        rdb_df[column] = pd.to_numeric(rdb_df[column], errors='coerce')

    return rdb_df


def plot_vrad_vs_rjd(obs_name):
    obs_path = file_path / obs_name
    rdb_df = read_rdb_file(obs_path)

    fig = go.Figure()
    fig.add_trace(
        go.Scatter(
            x=rdb_df['rjd'],
            y=rdb_df['vrad'],
            error_y=dict(type='data', array=rdb_df['svrad'], visible=True),
            mode='markers+lines',
            marker=dict(size=7),
            name=obs_name,
            text=rdb_df['file_rootpath'],
            hovertemplate='rjd=%{x}<br>vrad=%{y}<br>svrad=%{error_y.array}<br>file=%{text}<extra></extra>'
        )
    )

    fig.update_layout(
        title=f'vrad vs rjd: {obs_name}',
        xaxis_title='rjd',
        yaxis_title='vrad',
        template='plotly_white',
        hovermode='closest'
    )

    fig.show()

In [150]:
all_obs_names = sorted(path.name for path in file_path.glob('*.rdb'))
obs_selector = widgets.Dropdown(options=all_obs_names, description='obs_name:')
interactive_plot = widgets.interactive_output(plot_vrad_vs_rjd, {'obs_name': obs_selector})
display(obs_selector, interactive_plot)

Dropdown(description='obs_name:', options=('HD189733_esp19_3.rdb', 'HD189733_esp19_4.rdb', 'HD209458_esp19_1.r…

Output()

In [151]:
#obs_name = 'WASP62_esp19_1.rdb'


def fit_linear_rv(obs_df, fit_mask=None):
    clean_df = obs_df.dropna(subset=['rjd', 'vrad', 'svrad']).copy()
    if fit_mask is None:
        fit_mask = pd.Series(True, index=clean_df.index)
    else:
        fit_mask = pd.Series(fit_mask, index=clean_df.index).fillna(False)

    fit_df = clean_df.loc[fit_mask]
    rjd_ref = fit_df['rjd'].median()
    x_fit = fit_df['rjd'] - rjd_ref
    weights = 1 / fit_df['svrad']
    slope, intercept = np.polyfit(x_fit, fit_df['vrad'], deg=1, w=weights)

    model = intercept + slope * (clean_df['rjd'] - rjd_ref)
    result_df = clean_df.assign(
        fit_mask=fit_mask,
        linear_model=model,
        linear_residual=clean_df['vrad'] - model,
    )
    fit_result = {
        'rjd_ref': rjd_ref,
        'intercept': intercept,
        'slope': slope,
        'n_fit_points': int(fit_mask.sum()),
    }

    return result_df, fit_result


def iteratively_exclude_largest_residuals(obs_df, n_iterations=1, n_exclude_each=1):
    clean_df = obs_df.dropna(subset=['rjd', 'vrad', 'svrad']).copy()
    fit_mask = pd.Series(True, index=clean_df.index)
    history = []

    for iteration in range(n_iterations):
        fitted_df, fit_result = fit_linear_rv(clean_df, fit_mask=fit_mask)
        candidate_residuals = fitted_df.loc[fit_mask, 'linear_residual'].abs()
        exclude_indices = candidate_residuals.nlargest(n_exclude_each).index
        fit_mask.loc[exclude_indices] = False
        history.append(
            {
                'iteration': iteration + 1,
                **fit_result,
                'excluded_indices': list(exclude_indices),
                'excluded_files': fitted_df.loc[exclude_indices, 'file_rootpath'].tolist(),
                'excluded_rjd': fitted_df.loc[exclude_indices, 'rjd'].tolist(),
                'excluded_vrad': fitted_df.loc[exclude_indices, 'vrad'].tolist(),
                'excluded_residual': fitted_df.loc[exclude_indices, 'linear_residual'].tolist(),
            }
        )

    initial_df, initial_fit = fit_linear_rv(clean_df)
    final_df, final_fit = fit_linear_rv(clean_df, fit_mask=fit_mask)

    return initial_df, initial_fit, final_df, final_fit, pd.DataFrame(history)


def model_curve(obs_df, fit_result, n_points=300):
    model_rjd = np.linspace(obs_df['rjd'].min(), obs_df['rjd'].max(), n_points)
    model_vrad = fit_result['intercept'] + fit_result['slope'] * (model_rjd - fit_result['rjd_ref'])

    return model_rjd, model_vrad


def plot_iterative_linear_fit(obs_df, initial_fit, final_df, final_fit, obs_name):
    initial_model_rjd, initial_model_vrad = model_curve(obs_df, initial_fit)
    final_model_rjd, final_model_vrad = model_curve(obs_df, final_fit)
    kept_df = final_df.loc[final_df['fit_mask']]
    excluded_df = final_df.loc[~final_df['fit_mask']]

    fig = go.Figure()
    fig.add_trace(
        go.Scatter(
            x=kept_df['rjd'],
            y=kept_df['vrad'],
            error_y=dict(type='data', array=kept_df['svrad'], visible=True),
            mode='markers',
            marker=dict(size=7),
            name='Used RV points',
            text=kept_df['file_rootpath'],
            hovertemplate='rjd=%{x}<br>vrad=%{y}<br>svrad=%{error_y.array}<br>file=%{text}<extra></extra>',
        )
    )
    fig.add_trace(
        go.Scatter(
            x=excluded_df['rjd'],
            y=excluded_df['vrad'],
            error_y=dict(type='data', array=excluded_df['svrad'], visible=True),
            mode='markers',
            marker=dict(size=10, symbol='x'),
            name='Excluded largest residuals',
            text=excluded_df['file_rootpath'],
            hovertemplate='rjd=%{x}<br>vrad=%{y}<br>svrad=%{error_y.array}<br>file=%{text}<extra></extra>',
        )
    )
    fig.add_trace(
        go.Scatter(
            x=initial_model_rjd,
            y=initial_model_vrad,
            mode='lines',
            line=dict(width=2, dash='dash'),
            name='Initial linear fit',
            hovertemplate='rjd=%{x}<br>initial model=%{y}<extra></extra>',
        )
    )
    fig.add_trace(
        go.Scatter(
            x=final_model_rjd,
            y=final_model_vrad,
            mode='lines',
            line=dict(width=3),
            name='Refit after exclusions',
            hovertemplate='rjd=%{x}<br>refit model=%{y}<extra></extra>',
        )
    )

    fig.update_layout(
        title=f'Iterative linear RV fit: {obs_name}',
        xaxis_title='rjd',
        yaxis_title='vrad',
        template='plotly_white',
        hovermode='closest',
    )

    return fig


"""
obs_df = read_rdb_file(best_rm_path / obs_name)
initial_df, initial_fit, final_df, final_fit, exclusion_history = iteratively_exclude_largest_residuals(
    obs_df,
    n_iterations=12,
    n_exclude_each=1,
)

print(
    f"initial fit: vrad = {initial_fit['intercept']:.6f} "
    f"+ {initial_fit['slope']:.6f} * (rjd - {initial_fit['rjd_ref']:.8f})"
)
print(
    f"refit: vrad = {final_fit['intercept']:.6f} "
    f"+ {final_fit['slope']:.6f} * (rjd - {final_fit['rjd_ref']:.8f})"
)
display(exclusion_history)

fig = plot_iterative_linear_fit(obs_df, initial_fit, final_df, final_fit, obs_name)
fig.show()
"""

'\nobs_df = read_rdb_file(best_rm_path / obs_name)\ninitial_df, initial_fit, final_df, final_fit, exclusion_history = iteratively_exclude_largest_residuals(\n    obs_df,\n    n_iterations=12,\n    n_exclude_each=1,\n)\n\nprint(\n    f"initial fit: vrad = {initial_fit[\'intercept\']:.6f} "\n    f"+ {initial_fit[\'slope\']:.6f} * (rjd - {initial_fit[\'rjd_ref\']:.8f})"\n)\nprint(\n    f"refit: vrad = {final_fit[\'intercept\']:.6f} "\n    f"+ {final_fit[\'slope\']:.6f} * (rjd - {final_fit[\'rjd_ref\']:.8f})"\n)\ndisplay(exclusion_history)\n\nfig = plot_iterative_linear_fit(obs_df, initial_fit, final_df, final_fit, obs_name)\nfig.show()\n'

In [212]:
obs_name = 'WASP166_esp19_2.rdb'
obs_df = read_rdb_file(best_rm_path / obs_name)

def iteratively_clip_by_rms(obs_name, rms_threshold=2.0, max_iterations=10, observations_path=best_rm_path):
    obs_df = read_rdb_file(observations_path / obs_name)
    clean_df = obs_df.dropna(subset=['rjd', 'vrad', 'svrad']).copy()
    fit_mask = pd.Series(True, index=clean_df.index)
    history = []

    for iteration in range(max_iterations):
        fitted_df, fit_result = fit_linear_rv(clean_df, fit_mask=fit_mask)
        fit_residuals = fitted_df.loc[fit_mask, 'linear_residual']
        residual_rms = np.sqrt(np.mean(fit_residuals**2))
        clip_limit = rms_threshold * residual_rms
        exclude_mask = fit_mask & (fitted_df['linear_residual'].abs() > clip_limit)
        exclude_indices = fitted_df.index[exclude_mask]

        history.append(
            {
                'iteration': iteration + 1,
                **fit_result,
                'residual_rms': residual_rms,
                'clip_limit': clip_limit,
                'n_excluded': len(exclude_indices),
                'excluded_indices': list(exclude_indices),
                'excluded_files': fitted_df.loc[exclude_indices, 'file_rootpath'].tolist(),
                'excluded_rjd': fitted_df.loc[exclude_indices, 'rjd'].tolist(),
                'excluded_vrad': fitted_df.loc[exclude_indices, 'vrad'].tolist(),
                'excluded_residual': fitted_df.loc[exclude_indices, 'linear_residual'].tolist(),
            }
        )

        if len(exclude_indices) == 0:
            break

        fit_mask.loc[exclude_indices] = False

    initial_df, initial_fit = fit_linear_rv(clean_df)
    final_df, final_fit = fit_linear_rv(clean_df, fit_mask=fit_mask)

    return initial_df, initial_fit, final_df, final_fit, pd.DataFrame(history)


def fit_with_edge_points(obs_name, n_first=5, n_last=6, observations_path=best_rm_path):
    obs_df = read_rdb_file(observations_path / obs_name)
    clean_df = obs_df.dropna(subset=['rjd', 'vrad', 'svrad']).copy().sort_values('rjd')

    fit_mask = pd.Series(False, index=clean_df.index)
    fit_mask.loc[clean_df.head(n_first).index] = True
    fit_mask.loc[clean_df.tail(n_last).index] = True

    fitted_df, fit_result = fit_linear_rv(clean_df, fit_mask=fit_mask)
    fit_history = pd.DataFrame(
        [
            {
                **fit_result,
                'fit_method': 'edge_points',
                'n_first': n_first,
                'n_last': n_last,
                'fit_indices': list(clean_df.index[fit_mask]),
            }
        ]
    )

    return fitted_df, fit_result, fit_history


def plot_linear_fit_selection(obs_df, fitted_df, fit_result, obs_name, fit_label='Linear fit'):
    model_rjd, model_vrad = model_curve(obs_df, fit_result)
    fit_df = fitted_df.loc[fitted_df['fit_mask']]
    unused_df = fitted_df.loc[~fitted_df['fit_mask']]

    fig = go.Figure()
    fig.add_trace(
        go.Scatter(
            x=unused_df['rjd'],
            y=unused_df['vrad'],
            error_y=dict(type='data', array=unused_df['svrad'], visible=True),
            mode='markers',
            marker=dict(size=7, opacity=0.55),
            name='Not used in fit',
            text=unused_df['file_rootpath'],
            hovertemplate='rjd=%{x}<br>vrad=%{y}<br>svrad=%{error_y.array}<br>file=%{text}<extra></extra>',
        )
    )
    fig.add_trace(
        go.Scatter(
            x=fit_df['rjd'],
            y=fit_df['vrad'],
            error_y=dict(type='data', array=fit_df['svrad'], visible=True),
            mode='markers',
            marker=dict(size=10, symbol='circle-open'),
            name='Used in fit',
            text=fit_df['file_rootpath'],
            hovertemplate='rjd=%{x}<br>vrad=%{y}<br>svrad=%{error_y.array}<br>file=%{text}<extra></extra>',
        )
    )
    fig.add_trace(
        go.Scatter(
            x=model_rjd,
            y=model_vrad,
            mode='lines',
            line=dict(width=3),
            name=fit_label,
            hovertemplate='rjd=%{x}<br>model vrad=%{y}<extra></extra>',
        )
    )
    fig.update_layout(
        title=f'{fit_label}: {obs_name}',
        xaxis_title='rjd',
        yaxis_title='vrad',
        template='plotly_white',
        hovermode='closest',
    )

    return fig


fit_method_for_plot = 'edge_points'  # use 'rms_clip' or 'edge_points'

if fit_method_for_plot == 'rms_clip':
    rms_initial_df, rms_initial_fit, rms_final_df, rms_final_fit, rms_clip_history = iteratively_clip_by_rms(
        obs_name,
        rms_threshold=1.5,
        max_iterations=5,
    )
    print(
        f"RMS-clipped fit: vrad = {rms_final_fit['intercept']:.6f} "
        f"+ {rms_final_fit['slope']:.6f} * (rjd - {rms_final_fit['rjd_ref']:.8f})"
    )
    fig = plot_iterative_linear_fit(obs_df, rms_initial_fit, rms_final_df, rms_final_fit, obs_name)
    fig.update_layout(title=f'RMS-clipped linear RV fit: {obs_name}')
elif fit_method_for_plot == 'edge_points':
    edge_fitted_df, edge_fit, edge_fit_history = fit_with_edge_points(
        obs_name,
        n_first=20,
        n_last=20,
    )
    print(
        f"edge-points fit: vrad = {edge_fit['intercept']:.6f} "
        f"+ {edge_fit['slope']:.6f} * (rjd - {edge_fit['rjd_ref']:.8f})"
    )
    display(edge_fit_history)
    fig = plot_linear_fit_selection(obs_df, edge_fitted_df, edge_fit, obs_name, fit_label='Edge-points linear fit')
else:
    raise ValueError("fit_method_for_plot must be 'rms_clip' or 'edge_points'")

fig.show()

edge-points fit: vrad = 23505.685237 + -10.980870 * (rjd - 59264.72715448)


,rjd_ref,intercept,slope,n_fit_points,fit_method,n_first,n_last,fit_indices
0,59264.727154,23505.685237,-10.98087,40,edge_points,20,20,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,..."


In [213]:
def make_linear_corrected_observation_df(
    obs_name,
    fit_method='rms_clip',
    rms_threshold=1.5,
    max_iterations=7,
    n_first=6,
    n_last=5,
    observations_path=best_rm_path,
):
    if fit_method == 'rms_clip':
        _, _, fitted_df, fit_result, fit_history = iteratively_clip_by_rms(
            obs_name,
            rms_threshold=rms_threshold,
            max_iterations=max_iterations,
            observations_path=observations_path,
        )
    elif fit_method == 'edge_points':
        fitted_df, fit_result, fit_history = fit_with_edge_points(
            obs_name,
            n_first=n_first,
            n_last=n_last,
            observations_path=observations_path,
        )
    else:
        raise ValueError("fit_method must be 'rms_clip' or 'edge_points'")

    output_columns = ['rjd', 'vrad', 'fwhm', 'bis_span', 'contrast', 's_mw', 'ha', 'na', 'ca', 'rhk']
    for column in output_columns:
        if column not in fitted_df.columns:
            fitted_df[column] = np.nan

    corrected_df = fitted_df[output_columns].copy()
    corrected_df['true_vrad'] = fitted_df['linear_model']
    corrected_df['true_vrad'] = np.round(corrected_df['true_vrad'] - corrected_df['true_vrad'].mean(), decimals=2)
    corrected_df = corrected_df[
        ['rjd', 'vrad', 'true_vrad', 'fwhm', 'bis_span', 'contrast', 's_mw', 'ha', 'na', 'ca', 'rhk']
    ]

    output_path = observations_path / "linear_corrected"
    output_path.mkdir(exist_ok=True)
    corrected_file = output_path / f"{obs_name.replace('.rdb', '')}_linear_corrected.csv"
    corrected_df.to_csv(corrected_file, index=False)

    parameter_file = output_path / "linear_model_parameters.csv"
    parameter_row = pd.DataFrame(
        [
            {
                'observation_file': obs_name,
                'corrected_file': corrected_file.name,
                'fit_method': fit_method,
                'rms_threshold': rms_threshold if fit_method == 'rms_clip' else np.nan,
                'max_iterations': max_iterations if fit_method == 'rms_clip' else np.nan,
                'n_first': n_first if fit_method == 'edge_points' else np.nan,
                'n_last': n_last if fit_method == 'edge_points' else np.nan,
                'rjd_ref': fit_result['rjd_ref'],
                'intercept': fit_result['intercept'],
                'slope': fit_result['slope'],
                'n_fit_points': fit_result['n_fit_points'],
            }
        ]
    )
    parameter_row.to_csv(
        parameter_file,
        mode='a',
        header=not parameter_file.exists(),
        index=False,
    )

    return corrected_df, fit_result, fit_history, fitted_df


#obs_name = 'HD209458_esp19_1.rdb'

corrected_obs_df, corrected_fit, corrected_clip_history, corrected_fit_df = make_linear_corrected_observation_df(
    obs_name,
    fit_method='edge_points',
    rms_threshold=1.5,
    max_iterations=5,
    n_first=10,
    n_last=8,
)

print(
    f"linear correction fit: vrad = {corrected_fit['intercept']:.6f} "
    f"+ {corrected_fit['slope']:.6f} * (rjd - {corrected_fit['rjd_ref']:.8f})"
)
print(f"mean-centered linear model true_vrad mean = {corrected_obs_df['true_vrad'].mean():.6e}")
display(corrected_obs_df)

linear correction fit: vrad = 23506.799634 + -5.638991 * (rjd - 59264.61413622)
mean-centered linear model true_vrad mean = 0.000000e+00


,rjd,vrad,true_vrad,fwhm,bis_span,contrast,s_mw,ha,na,ca,rhk
0,59264.598231,23507.218946,0.73,9523.543980,-14.080011,49.661668,170.489404,0.195186,0.342988,0.133611,-4.844330
1,59264.600100,23507.250914,0.72,9512.134580,-13.561585,49.681032,174.196159,0.193679,0.342991,0.137161,-4.825502
2,59264.601936,23506.322919,0.71,9530.562878,-14.766767,49.656616,175.958244,0.193399,0.342377,0.138849,-4.816830
3,59264.603869,23508.035131,0.70,9520.393228,-14.459743,49.687982,175.356377,0.193633,0.341311,0.138272,-4.819772
4,59264.605705,23506.680885,0.68,9517.233652,-21.464382,49.692975,174.010275,0.193454,0.342585,0.136983,-4.826427
...,...,...,...,...,...,...,...,...,...,...,...
134,59264.848417,23505.190410,-0.68,9521.571397,-14.605047,49.636330,174.239702,0.193496,0.335811,0.137203,-4.825285
135,59264.850275,23507.521068,-0.69,9511.547939,-22.286498,49.685179,175.175108,0.193934,0.339455,0.138099,-4.820662
136,59264.852122,23502.074585,-0.70,9508.999924,-27.967058,49.666845,178.493363,0.195094,0.337571,0.141277,-4.804649
137,59264.853980,23505.787966,-0.72,9523.667773,-20.156412,49.675099,171.804416,0.194315,0.338996,0.134870,-4.837556


In [154]:
corrected_obs_df.describe()

,rjd,vrad,true_vrad,fwhm,bis_span,contrast,s_mw,ha,na,ca,rhk
count,62.000000,62.000000,62.000000,62.000000,62.000000,62.000000,62.000000,62.000000,62.000000,62.000000,62.000000
mean,59282.772736,-19721.453106,0.000323,8649.099326,7.216975,48.363632,161.899227,0.196736,0.385927,0.125382,-4.854890
std,0.075456,5.445021,1.662980,7.187041,4.506897,0.033772,1.957119,0.001132,0.001044,0.001875,0.011670
min,59282.645324,-19735.198095,-2.810000,8625.593490,-8.587065,48.307923,154.501576,0.193518,0.383557,0.118297,-4.900226
25%,59282.708986,-19724.798346,-1.407500,8644.122397,4.921863,48.341271,160.941728,0.196086,0.385322,0.124465,-4.860364
50%,59282.772768,-19720.620368,-0.005000,8649.400921,7.258166,48.361077,162.520677,0.196549,0.385939,0.125977,-4.851125
75%,59282.836485,-19717.456156,1.405000,8654.161907,10.073024,48.386017,163.044206,0.197642,0.386663,0.126479,-4.848104
max,59282.900289,-19713.229998,2.810000,8665.955120,16.782048,48.454909,164.426905,0.198789,0.388132,0.127804,-4.840226
